# LARA Training — Google Colab
Modifie `EXPERIMENT` ci-dessous puis exécute toutes les cellules.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────
EXPERIMENT   = "diff_attn"  # diff_attn | mor | coconut | lara_full | lara_v2_full | lara_v2_dca | lara_v2_rope
MAX_ITERS    = 5000
BATCH_SIZE   = 8
GRAD_ACCUM   = 16
N_EMBD       = 1024
N_LAYER      = 6
N_HEAD       = 8
BLOCK_SIZE   = 512
LR           = "3e-4"
WARMUP       = 500
N_RECURSIONS = 4
WANDB        = False
# ──────────────────────────────────────────────────────────────

RUN_NAMES = {
    'baseline':     'exp_a_baseline',
    'diff_attn':    'exp_b_diff_attn',
    'mor':          'exp_c_mor',
    'coconut':      'exp_d_coconut',
    'lara_full':    'exp_e_lara_full',
    'lara_v2':      'exp_f_lara_v2',
    'lara_v2_dca':  'exp_h_lara_v2_dca',
    'lara_v2_full': 'exp_g_lara_v2_full',
    'lara_v2_rope': 'exp_i_lara_v2_rope',
}
RUN_NAME = RUN_NAMES[EXPERIMENT]
print(f'Expérience : {EXPERIMENT} → checkpoint : {RUN_NAME}_best.pt')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/LARA_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints Drive : {DRIVE_DIR}')

In [ ]:
import os
if os.path.exists('/content/LARA'):
    !git -C /content/LARA pull
else:
    !git clone https://github.com/s3basti3nDev/LARA.git /content/LARA

if os.path.exists('/content/LARA/lara/train.py'):
    LARA_DIR = '/content/LARA/lara'
elif os.path.exists('/content/LARA/train.py'):
    LARA_DIR = '/content/LARA'
else:
    raise FileNotFoundError('train.py introuvable')

os.chdir(LARA_DIR)
print(f'Dossier : {LARA_DIR}')
!ls

In [ ]:
!pip install tiktoken datasets wandb -q
!nvidia-smi | grep -E 'GPU|Memory'

In [ ]:
import os
os.environ['PYTHONUNBUFFERED'] = '1'

args = (
    f"--experiment {EXPERIMENT} "
    f"--dataset fineweb "
    f"--n_embd {N_EMBD} --n_layer {N_LAYER} --n_head {N_HEAD} --block_size {BLOCK_SIZE} "
    f"--learning_rate {LR} --warmup_iters {WARMUP} "
    f"--batch_size {BATCH_SIZE} --grad_accum {GRAD_ACCUM} "
    f"--max_iters {MAX_ITERS} --device cuda --compile"
)
if EXPERIMENT in ('lara_v2', 'lara_v2_dca', 'lara_v2_full', 'lara_v2_rope'):
    args += f" --n_recursions {N_RECURSIONS}"
if WANDB:
    args += " --wandb"

print(f'Lancement : python -u train.py {args}')
!python -u train.py {args}

In [ ]:
# Copier le checkpoint vers Google Drive
import shutil, os
src = f'checkpoints/{RUN_NAME}_best.pt'
dst = f'{DRIVE_DIR}/{RUN_NAME}_best.pt'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Checkpoint sauvegardé : {dst}')
else:
    print('Checkpoints disponibles :', os.listdir('checkpoints') if os.path.exists('checkpoints') else 'aucun')

In [ ]:
!python -u evaluate.py --model {dst} --dataset fineweb